# Lab Exercise: Detecting Outdated Orthophotos via AI Segmentation & Overture Maps

**📝 Project Brief.** The City of Graz wants to systematically identify areas where their official orthophotos (e.g., from 2016-2018) are severely outdated due to recent urban development. Instead of manually scanning the whole city, they want an automated pipeline that compares **historical imagery** with **current map data** (Overture Maps) to flag "hotspots of change", and help prioritise areas for future aerial re-survey.

> [!Note]
> We do **not** prove where the orthophoto is outdated.  
> We identify places where **historical imagery and current building data disagree strongly enough to justify review**.

**🚚 What you will deliver**
1. A reproducible notebook that:
   - downloads historical orthophotos for a specific Region of Interest via the ArcGIS REST API
   - queries current building footprints from **Overture Maps** using DuckDB
   - runs an AI foundation model (**SamGeo**:Segment Anything Model) to extract buildings from the old orthophoto
   - computes **inconsistency indicators**
   - produces a Plotly map highlighting **hotspots of disagreement**

2. A short advisory brief (<500 words) answering:
   - **Top 3 sub-regions to flag** for new aerial surveying based on your metrics
   - **Technical risks** of using AI segmentation (SamGeo) for official building registries
   - **Ethical & Open Data considerations** when mixing Overture data with state-owned imagery


<div class="alert alert-warning">

#### 📃 Data sources & licences (you must always acknowledge them)
- **Overture Maps** data is licensed under the Community Data License Agreement (CDLA). [👀 see](https://overturemaps.org/)
- **Orthofotos Land Steiermark** are provided as Open Government Data (OGD) under CC BY 4.0. [👀 see](https://data.steiermark.at/)

#### 📊 Analytical warning
In this exercise, **Overture is a current reference dataset, not ground truth for the historical image**.  

</div>

---

## 🎯 Learning outcomes

---

1. **Cloud-Native Geoprocessing:** Query massive remote datasets (Overture GeoParquet) efficiently using `DuckDB` spatial extensions.
2. **GeoAI Application:** Apply Meta's Segment Anything Model (`SamGeo`) to georeferenced raster data to extract vector building footprints.
3. **Spatial Metrics:** Design and calculate computer vision metrics like **Intersection over Union (IoU)** using `GeoPandas`.
4. **CRS Mastery:** Seamlessly transition between web-mercator (image API), WGS84 (Overture/Plotly), and local projected CRS (metric area calculations).
5. **Interactive Visualization:** Overlay raster and vector data in a web-friendly `Plotly` map.


---

## 0. 🔧 Setup & Configuration

---


In [ ]:
# installs for the notebook, uncomment to install them if not done already, or select the library you need to install

#%uv pip install geopandas duckdb shapely requests rasterio plotly segment-geospatial leafmap localtileserver geoai-py

In [ ]:
# system
import os
from pathlib import Path

# Clear conflicting PROJ environment variables if they exist
# (I had to do this so that my leafmap works)
os.environ.pop('PROJ_LIB', None)
os.environ.pop('PROJ_DATA', None)

# data handling
import requests
import duckdb
import geopandas as gpd

# geo
import rasterio
from shapely.geometry import box
import shapely
from samgeo import SamGeo
from rasterio.mask import mask

# visualization
import leafmap
import plotly.express as px



### 0.1 Project structure and configuration

<div class="alert alert-info">  
First, we need to set up our project workspace and define our Area of Interest (AOI). 

* **CRS (Coordinate Reference Systems):** We define WGS84 for downloading web data and UTM Zone 33N for accurate metric calculations later on.
* **AOI (Area of Interest):** We use a bounding box (bbox) to define which part of Graz we want to analyze.
</div>

<div class="alert alert-danger">

**🚀 TODO: Choose an AOI**

1. Go to http://bboxfinder.com
2. Zoom to Graz and draw a SQUARE ☐ bounding box
3. RULE: The area should be ~15-20 Hectares!
4. Copy the coordinates from the bottom left (Format: min_lon, min_lat, max_lon, max_lat) into the List "bbox_custom"
5. Set `ACTIVE_BBOX` so the rest of the notebook uses your choice
</div>

In [ ]:
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
CONFIG = {
    "crs_geographic": "EPSG:4326",   # WGS84
    "crs_projected": "EPSG:32633",   # UTM Zone 33N for metric area calculations
    "ortho_base_url": "https://gis.stmk.gv.at/image/rest/services/OGD_DOP/Flug_2016_2018_RGB/ImageServer/exportImage",
    
    "bbox_custom": [None, None, None, None],

    # Example bounding boxes for different districts in Graz (you can use these if you don't want to draw your own)
    "bbox_waltendorf": [15.475338,47.060088,15.495937,47.066855],
    "bbox_goesting": [15.387114,47.092339,15.407714,47.099103],
    "bbox_mariatrost": [15.443537,47.090659,15.464137,47.097423],
    "bbox_liebenau": [15.457131,47.032219,15.477548,47.038990],
    "bbox_graz_center": [15.436778,47.067842,15.443537,47.072168]
}

In [ ]:
# Decide which one to use for the rest of the script:
ACTIVE_BBOX = # TODO Select the bbox

<div class="alert alert-success"> 

**🧠 Questions**: 
- Why is it useful to keep filenames, CRS, and AOI settings in `CONFIG`?
- Why do we use a **small AOI** here instead of testing the whole city at once?

---

## 2. 🏞️ Fetching Historical Imagery (ArcGIS REST API)

---


<div class="alert alert-info">  

Now that we have our bounding box, we need the actual aerial imagery. We will download a high-resolution historical Orthophoto (flown between 2016 and 2018) from the official Styrian GIS server (GIS-Steiermark).

The server uses the **ArcGIS REST API**. Instead of clicking "download" on a website, we will write a Python script that sends an HTTP `GET` request to the server, asking for a specific image based on our bounding box coordinates.


**💡 API Context:**
Servers have rules. If you ask for the whole city of Graz in full resolution, the server will block you or crash. We must use specific **URL parameters** to tell the server exactly what we want (Bounding Box, Format, Resolution...).

**Why this matters**
A scripted download is slower to design than manual clicking, but much better for:

- reproducibility,
- rerunning the workflow later,
- documenting exactly what data were requested.
</div>

<div class="alert alert-danger"> 

**🚀 TODO: Crack the API Parameters!**

We have provided the structure of the `download_orthophoto` function, but it is missing the correct parameters to talk to the ArcGIS server. 

1. Go to the [GIS Steiermark Image Server](https://gis.stmk.gv.at/image/rest/services/OGD_DOP/Flug_2016_2018_RGB/ImageServer).
2. Look at the **"Export Image"** page (at the very bottom).
3. Fill in the form (Bounding Box, Bounding Box Spatial Reference, Image Size, Image Format)
4. Click on "Export Image (GET)". The information you need can now be found in the URL on top!
5. Your task is to fill in the empty keys in the `params` dictionary in the code block below. 
6. We will tell the server:
   * What **bounding box** to use.
   * That our bounding box is in **WGS84 (EPSG: 4326)** coordinates.
   * The **size** of the image
   * To send the image in **tiff** format.
   * To return the actual **image file**, not just an HTML.

*Hint: Look closely at the parameter names expected by the API in the URL!*
</div>

<div class="alert alert-info">

**💡 Python Tips: Dictionaries (Keys & Values)**

A Python **Dictionary** (`dict`) is used to store data values in **key-value pairs**. You can think of the **key** as the label or variable name, and the **value** as the actual data attached to it.

- **Syntax:** Dictionaries are written with curly brackets `{}`. The key and value are separated by a colon `:`, and each pair is separated by a comma.
    ```python
    my_settings = {
        "color": "blue",    # "color" is the key, "blue" is the value
        "opacity": 0.5
    }
    ```

- **Why use them for APIs?** When making HTTP requests, web servers expect specific parameter names. The Python `requests` library is smart: it takes your dictionary and automatically translates the keys and values into a URL query string (for example, `?color=blue&opacity=0.5`). 

- **Your Task:** In the code below, the *values* are already set. Your job is to replace the `?` with the exact *keys* (parameter names) that the ArcGIS API requires to understand our request!
</div>

In [ ]:
# Orthophoto Download

def download_orthophoto(bbox: list, url: str, output_path: Path, size: str = "2000,2000") -> None: 
    """
    Downloads a historical orthophoto from the ArcGIS REST API.
    
    Args:
        bbox (list): Bounding box coordinates in format [min_lon, min_lat, max_lon, max_lat] defining the area to download imagery for
        url (str): Base URL of the ArcGIS ImageServer REST API endpoint
        output_path (Path): File path where the downloaded GeoTIFF will be saved
        size (str, optional): Image dimensions in format "width,height". Defaults to "2000,2000" for a square 2000x2000 pixel image
    """
    # Convert the bbox list to a comma-separated string
    bbox_str = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
    
    # Define the exact parameters required by the ArcGIS ImageServer
    params = { 
        #TODO: Change the keys to the correct ones for the ArcGIS ImageServer
        "?": bbox_str,   # The bounding box in the format: min_lon,min_lat,max_lon,max_lat
        "?": "4326",      # Explicitly state our bbox is in WGS84
        "?": size,          # Constrain image size to avoid server limits
        "?": "tiff",      # Request GeoTIFF format
        "?": "image"           # Request the actual image file, not JSON metadata

    }
    
    print(f"Requesting imagery from {url} with parameters: {params}")
    response = requests.get(url, params=params)
    
    # Raise an exception if the HTTP request failed (e.g., 404 or 400 error)
    response.raise_for_status()
    
    # Write the binary content to our data directory
    with open(output_path, "wb") as f:
        f.write(response.content)
        
    print(f"Orthophoto saved to: {output_path}")

In [ ]:
# Execute the download for our Graz bounding box
ortho_file = DATA_DIR / # TODO Name of your orthophoto file
download_orthophoto(
    # TODO add the arguments of the function    
)

---

## 3. ✂️ Clipping the Image (Rasterio)

---


<div class="alert alert-info">  

Because we requested a fixed `2000,2000` pixel image, but your custom bounding box is likely not a perfect square, the ArcGIS server automatically expanded the geographical footprint of your image to make it square. 

If we don't fix this, our AI will analyze areas outside of our target zone!

To solve this, we will use `rasterio` (the standard Python library for raster data) and `shapely` (for geometry) to cleanly "clip" the downloaded GeoTIFF exactly to your original `ACTIVE_BBOX` coordinates.
</div>

<div class="alert alert-danger"> 

**🚀 TODO: Code Review**

Read the `clip_raster_to_bbox` function below. You do not need to write any Python code, but you need to understand the logic of **Step 3**. Do not shy away from using LLMs to explain code to you, as they are helpful tools for code comprehension. Just remember to ask them "Why is this line necessary?" instead of "Give me the answer!".

Look at the line: `gdf_bbox_projected = gdf_bbox.to_crs(src.crs)`

We defined our bounding box in WGS84 (EPSG:4326). In the previous step, we explicitly asked the ArcGIS server to send us the image in WGS84. Theoretically, both datasets already are in the same coordinate system.

</div>

In [ ]:
def clip_raster_to_bbox(input_raster_path, output_raster_path, bbox_wgs84):
    """
    Clips a raster image to a specific bounding box and saves the result.
    
    Args:
        input_raster_path (str or Path): Path to the input GeoTIFF raster file
        output_raster_path (str or Path): Path where the clipped raster will be saved
        bbox_wgs84 (list): Bounding box coordinates in WGS84 (EPSG:4326) format [min_lon, min_lat, max_lon, max_lat] defining the clipping extent
    """
    # 1. Create the Polygon from our bounding box
    min_lon, min_lat, max_lon, max_lat = bbox_wgs84
    bbox_polygon = box(min_lon, min_lat, max_lon, max_lat)
    
    # 2. Put it in a GeoDataFrame
    gdf_bbox = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs="EPSG:4326")
    
    # 3. Open the image, match the Coordinate Systems, and cut!
    with rasterio.open(input_raster_path) as src:
        # Reproject our polygon to match the image's exact CRS
        gdf_bbox_projected = gdf_bbox.to_crs(src.crs)
        projected_polygon = gdf_bbox_projected.geometry.iloc[0]
        
        # Cut the image (returns the new pixels and the new spatial mapping)
        out_image, out_transform = mask(src, [projected_polygon], crop=True)
        
        # Copy old metadata and update it with the new dimensions
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

    # 4. Save the cleanly clipped image
    with rasterio.open(output_raster_path, "w", **out_meta) as dest:
        dest.write(out_image)

In [ ]:
# --- MAIN SCRIPT  ---

clipped_ortho_file = DATA_DIR / # TODO Name of your orthophoto file

# We call our function with our inputs
clip_raster_to_bbox(
    # TODO add the arguments of the function  
)

print(f"Clipped image saved to: {clipped_ortho_file}")

<div class="alert alert-success"> 

**🧠 Questions**: 
- What could go wrong if you skip the clipping step?
- If we know both are in WGS84, why is writing `to_crs(src.crs)` dynamically still a highly recommended **Software Engineering Best Practice** instead of just hardcoding it or skipping the step entirely? 
- Write a short, 1-2 sentence comment at the `TODO` marker explaining your thoughts.

In [ ]:
# TODO: Explain why '.to_crs(src.crs)' is a good practice here.
#
# YOUR EXPLANATION: 
#
#

---

## 4. 🏢 Query Current Buildings via DuckDB (Overture Maps)

---


<div class="alert alert-info">  
We have our historical imagery. Now we need the modern "Ground Truth" to compare it against. We will use Overture Maps, an open dataset created by Amazon, Meta, Microsoft, and TomTom, which contains highly accurate building footprints.

Instead of downloading gigabytes of data, we will use **DuckDB**, a fast analytical database. DuckDB allows us to write an SQL query that pulls *only* the buildings within our specific bounding box directly from Overture's cloud storage (Amazon S3), without downloading the rest of the world!
</div>

<div class="alert alert-danger"> 

**🚀 TODO: Cloud SQL Querying**

Your task is to complete the `fetch_overture_buildings` function.

1. **Unpack the coordinates:** Extract `min_lon`, `min_lat`, `max_lon`, and `max_lat` from the `bbox` list.
2. **Load Database Extensions:** DuckDB needs specific extensions to handle geographic data and download from the web. You need to `INSTALL` and `LOAD` the `spatial` and `httpfs` extensions. (The S3 region is already set for you).
3. **Complete the SQL Query:** Check the Overture schema documentation to find the theme and type for buildings to fix the S3 path, and insert your unpacked bounding box variables into the WHERE clause

**Hint: Look at the Overture Maps documentation:**
- [getting data](https://docs.overturemaps.org/getting-data/)
- [building schema](https://docs.overturemaps.org/schema/reference/buildings/building/)
</div>

In [ ]:
def fetch_overture_buildings(bbox: list, crs: str) -> gpd.GeoDataFrame:
    """
    Fetches current building footprints from Overture Maps for a specified bounding box.
    
    Args:
        bbox (list): Bounding box coordinates in format [min_lon, min_lat, max_lon, max_lat] defining the area to search for buildings in
        crs (str): Coordinate reference system identifier (e.g., "EPSG:4326") for the returned GeoDataFrame
        
    Technical Notes:
        - Requires DuckDB extensions: 'spatial' for geographic operations and 'httpfs' for S3 access
        - Uses Overture Maps 2026-02-18.0 release data
        - Buildings are filtered to only include those completely within the bounding box
        - Geometry is returned as WKT (Well-Known Text) and converted to Shapely objects

    """

    # TASK 1: Unpack the coordinates
    # TODO: Extract the min and max longitude and latitude from the bbox list
    min_lon, min_lat, max_lon, max_lat = [None, None, None, None] #Change this!
    
    # 1. Connect to DuckDB and load extensions
    con = duckdb.connect()

    # TASK 2: Load Database Extensions
    # TODO: Write the SQL commands to INSTALL and LOAD 'spatial' and 'httpfs'

    con.execute("") # Install and load spatial extension
    con.execute("") # Install and load httpfs extension

    con.execute("SET s3_region='us-west-2';") # Sets the Amazon S3 region (Required by Overture)
    
    
    # TASK 3: Complete the SQL Query
    # TODO: Enter the correct Bounding Box variables into the SQL query below
    # Find the correct "theme" and "type" for buildings in the Overture Maps schema documentation
    query = f""" 
    SELECT
        id,
        type,
        names.primary AS name,
        ST_AsText(geometry) AS geometry
    FROM read_parquet('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=?/type=?/*', hive_partitioning=1)
    WHERE bbox.xmin >= {} -- TODO: Replace with the good value
      AND bbox.xmax <= {} -- TODO: Replace with the good value
      AND bbox.ymin >= {} -- TODO: Replace with the good value
      AND bbox.ymax <= {} -- TODO: Replace with the good value
    ;
    """
    
    # 3. Execute query and fetch as DataFrame
    df = con.execute(query).df()
    
    # 4. Convert WKT to Shapely geometries and build GeoDataFrame
    df['geometry'] = df['geometry'].apply(shapely.wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=crs)
    
    return gdf


In [ ]:
# Execute the function for our active bounding box
gdf_overture = fetch_overture_buildings(
    # TODO add the arguments of the function 
)
print(f"Loaded {len(gdf_overture)} buildings from Overture Maps.")

In [ ]:
gdf_overture.head()

In [ ]:
# TODO plot the result! 

<div class="alert alert-success"> 

**🧠 Questions**: 
- How to verify if you downloaded the right area?

<div class="alert alert-danger"> 

**🚀 TODO: Create a checkpoint of your data**

- Save the overture buildings data in a file
- try different formats (geojson, geoparquet, csv, geopackage, shapefile, etc.). Use the [geopandas documentation](https://geopandas.org/en/latest/docs/reference/api/geopandas.GeoDataFrame.to_file.html).
- Compare the file sizes and inspect what information is preserved or lost in each format. 

In [ ]:
overture_file = DATA_DIR / # TODO Filename of your data
gdf_overture.to_file(
    overture_file, # TODO to fill
)

<div class="alert alert-success"> 

**🧠 Questions**: 
- Which format is best for storage, analysis, sharing, and human readability?
- Which format would you send to a colleague using Python, to a web developer, and to someone using Excel?
- Can this vector dataset be saved as CSV, XYZ, or TIFF? If yes, under what conditions? If not, why not?

---

## 5. First visual comparison: historical image vs current Overture buildings (2016 vs. 2026)

---

<div class="alert alert-info">  


We now have two distinct datasets in our workspace:
1. `*.tif`: A historical satellite image from 2016-2018 (Raster/Pixels).
2. `gdf_overture`: Modern building footprints from 2026 (Vector/Polygons).

Before we let our AI extract the historical buildings, let's visually overlay the modern Overture data on top of the old orthophoto. We will use `leafmap`, a powerful geospatial visualization library in Python, to create an interactive web map right inside our notebook.
</div>

<div class="alert alert-danger"> 

**🚀 TODO: Map Initialization and Layer Stacking**

Your task is to build the interactive map by telling `leafmap` where to look and what to draw.

1. **Calculate the Center:** A map needs a starting camera position (`center_lat`, `center_lon`). Calculate the central point of our bounding box using basic math (average of the minimum and maximum coordinates). What other method could you use?
2. **Add the Raster Layer:** Pass the correct file path variable into `m.add_raster(...)` so the map loads your clipped orthophoto.
3. **Add the Vector Layer:** Pass the correct variable name into `m.add_gdf(...)` so the map displays the Overture buildings on top.
4. **Inspect the map:** What do you see? Are the buildings and the orthophoto aligned?

</div>

In [ ]:
''' 
VISUALIZATION OF OVERTURE BUILDINGS ON TOP OF ORTHOPHOTO (Leafmap)
'''

# TASK 1: Calculate the Center
overture_path = DATA_DIR / # TODO your exported file
gdf_overture = gpd.read_file(overture_path)

min_lon, min_lat, max_lon, max_lat = ACTIVE_BBOX

# TODO: Calculate the average of min and max coordinates to find the center
center_lat = None
center_lon = None

# Initialize the map camera
m = leafmap.Map(center=[center_lat, center_lon], zoom=16, height="600px")
m.clear_layers()

In [ ]:
ortho_path = str(DATA_DIR / # TODO your orthophoto file)    

# TASK 2: Add the Raster Layer
# TODO: Pass the path variable into the function below and give your layer a name and set opacity
m.add_raster(source=None, layer_name="", opacity=1)

# TASK 3: Add the Overture buildings
m.add_gdf(
    # TODO add your gdf data
    layer_name="Overture Buildings (2026)", 
    fill_colors=["#00a8ff"],
    color="blue",
    weight=1
)

In [ ]:
m # vizualize the map

<div class="alert alert-success"> 

**🧠 Questions**: 
- Do the Overture buildings align well with the roofs in the historical image?
- Which mismatches already look suspicious before using AI?

---

## 6. Unleashing the AI: Segment historical buildings with SAMGeo

---


<div class="alert alert-info">

Now it is time for the heavy lifting! We will use Meta's **Segment Anything Model (SAM)** to automatically detect and outline every distinct object in our historical orthophoto. 

We will use the standard `vit_b` (Vision Transformer - Base) model. It requires about 300MB of memory and runs efficiently on most standard laptops.


**🧠 How it works:**
In `automatic` mode, the AI drops a grid of points across the entire image. For every single point, it tries to grow a mask to find the boundaries of the object underneath it. Once it has all the masks (pixels), we vectorize them into polygons.

**🚧 Be careful with your interpretation:**
- SAMGeo is a **general segmentation model**
- it does **not** understand municipal building rules
- it may segment roofs, sheds, shadows, or other objects that only look building-like

</div>

<div class="alert alert-danger"> 

**🚀 TODO: Configure and Clean the AI Pipeline**

Your task is to set up the AI's parameters and clean the resulting data.

1. **File Management:** Choose fitting filenames for your outputs. We need one `.tif` file for the raw pixel masks, and one `.gpkg` file for the final vector polygons. 
2. **API Parameters:** Look at the [`samgeo` documentation](https://samgeo.gishub.org/examples/automatic_mask_generator/). Scroll down to the "Automatic mask generation options". We want to configure two things:
   * A parameter that controls the grid density (set the value to `32`).
   * A parameter that removes tiny noise fragments (set the value to `100`).
   Find the exact **parameter names** (the string keys in the dictionary) from the documentation example and replace the `sam_kwargs` dictionary in the code!
3. **Geometry Cleaning:** AI isn't perfect. Sometimes the vectorization creates broken or "empty" polygons. Use the GeoPandas properties (`.is_valid` and `.is_empty`) to filter out the garbage.
</div>

In [ ]:
# Paths for our input and outputs
ortho_file = DATA_DIR / "graz_ortho_clipped.tif"

# TASK 1: File Management
mask_raster_file = OUTPUT_DIR / # TODO fill filename raster mask
vector_file = OUTPUT_DIR / # TODO fill filename vector outputs

# TASK 2: API Parameters
# TODO: Look at the documentation link above. Find the correct parameter names (keys).
# We want to set the grid density to 32, and the minimum area to 20.
sam = SamGeo( # This line init the mode
    model_type="", # To fill
    automatic=True,
    sam_kwargs={
        "": None, # grid density
        "": None # minimum region area
    }
)


In [ ]:
# Generate raster masks from the orthophoto
print("Running AI segmentation on the orthophoto (This may take a few minutes!)")
sam.generate(
    source=str(ortho_file),
    output=str(mask_raster_file)
)

# Convert the raster masks into vector polygons
sam.tiff_to_vector(str(mask_raster_file), str(vector_file))

# Load the generated polygons into a GeoDataFrame and clean them
gdf_samgeo = gpd.read_file(vector_file)

In [ ]:
# TASK 3: Geometry Cleaning
# TODO: Filter the GeoDataFrame to KEEP valid shapes and DROP empty shapes
gdf_filter = # TODO filter out the geometries that are not valid, empty or NaN 
gdf_samgeo = gdf_samgeo[gdf_filter]
# additionally before filtering out the non valid geometry, you can try to make them valid using make_valid or buffer(0)

# Assign the correct coordinate system (WGS84)
gdf_samgeo.set_crs(CONFIG["crs_geographic"], inplace=True, allow_override=True)

In [ ]:
print(f"SAM generated: {len(gdf_samgeo)} valid polygons.")

# Display the first few rows
gdf_samgeo.head()

---

## 7. Visualizing the SAMGeo's Output

---

<div class="alert alert-info">


Our AI has finished processing the historical image and we have saved the resulting building footprints as a vector file. 

Before we do any mathematical comparisons, we need to look at what SAM actually found. We will build another interactive map, just like we did in Step 5, but this time we will overlay the **Historical AI Polygons** on top of the **Historical Orthophoto**.

SAMGeo is not the only AI segmentation model that exisit. At the end of the notebook, in the annex section, you can find an implementation of an other model, namely [GeoAI](https://opengeoai.org/). 
</div>

<div class="alert alert-danger"> 

**🚀 TODO: Build the Map**

You have already built an interactive map in **Step 5**. Use your code from that step as a blueprint to build this one!

1. **Load the Data:** Define the path to your newly generated SAM vector file (make sure the filename matches what you chose in Step 6!) and load it with GeoPandas.
2. **Calculate the Center:** Calculate `center_lat` and `center_lon` from the `ACTIVE_BBOX`.
3. **Initialize the Map:** Create a `leafmap.Map` and clear the default layers.
4. **Add the Layers:** * First, add the clipped orthophoto (`add_raster`).
   * Second, add the SAM AI Polygons (`add_gdf`). *Hint: Use a bright, contrasting color like `"magenta"` (`"#ff00ff"`) so the AI predictions stand out against the grey roofs!*
5. **Display:** Show the map.
</div>

In [ ]:
''' 
VISUALIZATION OF SAMGEO RESULTS ON TOP OF ORTHOPHOTO (Leafmap)
'''


# TASK: Recreate the leafmap Visualization
# Look at your code from Step 5 if you get stuck!


# 1. Define paths
ortho_file = str(DATA_DIR / "*.tif")
vector_file = str(OUTPUT_DIR / "*.*") # <-- TODO: Insert your chosen filename from Step 6!

# 2. Load the SAM polygons
gdf_samgeo = None # <-- TODO: Use GeoPandas to read the vector_file


# 3. Calculate Map Center
min_lon, min_lat, max_lon, max_lat = ACTIVE_BBOX
center_lon = None # <-- TODO
center_lat = None # <-- TODO


In [ ]:
# 4. Initialize Map
m = None # <-- TODO: Create the leafmap.Map (zoom=16, height="650px") and clear layers


# 5. Add Raster (Background)
# TODO: Add the ortho_file to the map 'm'


# 6. Add Vector (Foreground)
# TODO: Add the gdf_samgeo to the map 'm'


# Display the map
# TODO: Call your map variable to show it

<div class="alert alert-success"> 

**🧠 Questions**: 
- How does the result reflect the reality? How can we improve the results?
- Which kinds of roofs or urban objects are most likely to confuse SAMGeo?
- What is the trade-off when you change the minimum region area?

---

## 8. Urban Change Detection: SAMGeo vs. Overture

---


<div class="alert alert-info">

It's time to visually compare our extracted historical buildings (2016) against the modern Overture data (2026) using [Plotly](https://plotly.com/python/?_gl=1*128mdfk*_gcl_au*MTgyNDMyNjUxNi4xNzczMDUzMDcz*_ga*NTIzMjA2MjE2LjE3NzMwNTMwNzM.*_ga_6G7EE0JNSC*czE3NzYzNDgyMjEkbzUkZzAkdDE3NzYzNDgyMjEkajYwJGwwJGgw). 

</div>

<div class="alert alert-danger"> 

**🚀 TODO: Prepare Data and Build the Dashboard**

Your task is to prepare the data for Plotly and configure the interactive map.

1. **Reproject the Data:** Use the `.to_crs()` function to ensure both DataFrames are strictly in WGS84 format before plotting.
2. **Combine the Data:** We have added a "Source" column to both datasets to identify them. Use the pandas `pd.concat()` function to merge both DataFrames into one large table.
</div>

In [ ]:
print("Preparing data for the final visual comparison...")

# 1. Load the data 
samgeo_polygons_file = OUTPUT_DIR / "" # TODO: Insert the filename of your SAMGeo output from Step 6
overture_file = DATA_DIR / "" # TODO: Insert the filename of your Overture output from Step 4

gdf_sam = gpd.read_file(samgeo_polygons_file)
gdf_overture = gpd.read_file(overture_file)

# 2. Reprojection
# TODO: Ensure both DataFrames are using standard web coordinates
gdf_sam_wgs = gdf_sam.to_crs("CHANGEME")   #Insert the correct EPSG Codes    
gdf_overture_wgs = gdf_overture.to_crs("CHANGEME") 

# Add identification labels so Plotly knows which polygon belongs to which era
gdf_sam_wgs['Source'] = 'SAM GeoAI'
gdf_overture_wgs['Source'] = 'Overture'

# 3. Combine the Datasets into one large table
# TODO: Use Pandas to concatenate (pd.concat) both WGS84 DataFrames into a single one, remember to use ignore_index=True
gdf_combined = "?"

print("Building Plotly Map...")
# Calculate center for the camera
center_lat = gdf_combined.geometry.centroid.y.mean()
center_lon = gdf_combined.geometry.centroid.x.mean()


In [ ]:
# 4. Configure Plotly
fig = px.choropleth_mapbox(
    data_frame=gdf_combined,                   
    geojson=gdf_combined.geometry,                      
    locations=gdf_combined.index,      # Use the DataFrame index to match geometries
    color='Source',                    # Color the polygons based on the Source column               
    color_discrete_map={
        'Overture': '#00a8ff', # Bright Blue
        'SAM GeoAI': '#ff00ff'    # Magenta
    },
    center={"lat": center_lat, "lon": center_lon},
    mapbox_style="carto-darkmatter",
    zoom=15,
    opacity=0.6,
    hover_name='Source'
)

# Clean up layout
fig.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0},
    title_text="Urban Development Comparison (Click Legend to Toggle Layers)",
    legend_title_text="Datasets"
)

# Show the map
fig.show()

<div class="alert alert-success"> 

**🧠 Questions**: 
- Which mismatches look like likely real change?
- Which mismatches look more like segmentation or alignment problems?
- What can you see visually that a single overlap metric may not capture well?

---

## 9. Quantitative Evaluation

---

<div class="alert alert-info">
Visual comparisons are great, but Data Science requires hard numbers. How well did our AI perform in 2016 compared to the modern Open Data from 2026? 

Because we are dealing with spatial data (polygons), we calculate **Area-based Metrics**:
* **Precision:** Out of all the area the AI *thought* was a building, how much of it actually *is* a building in the Overture dataset?
* **Recall:** Out of all the true Overture building area, how much did our AI successfully *find*?
* **F1-Score:** The harmonic mean of Precision and Recall (Overall performance).

Finally, we will calculate the **Mean Centroid Distance**. This tells us, on average, how many meters the center of an AI-predicted building is shifted away from the Overture building's center.
</div>

<div class="alert alert-danger"> 

**🚀 TODO: Calculate Spatial Accuracy**

Your task is to finalize the spatial metrics calculation.

1. **Reprojection:** WGS84 calculates in "Degrees", which are useless for metric Area and Distance. Reproject both DataFrames to our projected CRS. *Hint: Do not type the EPSG code manually! Retrieve it from the `CONFIG` dictionary we set up in Step 1!*
2. **Calculate Total Area:** Find out how to calculate the total area in square meters. *Hint: Access the `geometry` property, calculate the `area`, and use a standard pandas function to `sum` it all up.*
3. **Calculate Centroids:** Find the exact center point of every building in both datasets. Replace the `CHANGEME` strings with the correct GeoPandas property.
4. **Calculate Mean Distance:** We use a powerful function called `sjoin_nearest` to link every SAM centroid to its closest Overture centroid. Your job is to calculate the **average** of the resulting `distance` column.
</div>

In [ ]:
print("Calculating quantitative metrics (SAMGeo vs. Overture)...")

gdf_sam = gpd.read_file(OUTPUT_DIR / "CHANGEME")
gdf_overture = gpd.read_file(DATA_DIR / "CHANGEME")


# TASK 1: Reprojection
# TODO: Use the CONFIG dictionary to pass the projected CRS (UTM Zone 33N)
gdf_sam_utm = gdf_sam.to_crs(None)       # <-- CHANGEME
gdf_ov_utm = gdf_overture.to_crs(None)   # <-- CHANGEME

# TASK 2: Calculate Total Area
# TODO: Calculate the area for the geometries and sum them up
sam_total_area = None # <-- CHANGEME
ov_total_area = None  # <-- CHANGEME

# Calculate the Intersection (Overlap) Area (Pre-filled for you)
# gpd.overlay does a precise polygon-on-polygon intersection
intersection = gpd.overlay(gdf_sam_utm, gdf_ov_utm, how='intersection')
intersection_area = intersection.geometry.area.sum()

# Calculate Binary Classification Metrics (consistency metric)
precision = intersection_area / sam_total_area if sam_total_area > 0 else 0
recall = intersection_area / ov_total_area if ov_total_area > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# # Disagreement layers
sam_only = gpd.overlay(gdf_sam_utm, gdf_ov_utm, how="difference")
overture_only = gpd.overlay(gdf_ov_utm, gdf_sam_utm, how="difference")

# TODO: Calculate the total area of the disagreement layers
sam_only_area = None # <-- CHANGEME
overture_only_area = None  # <-- CHANGEME

In [ ]:
# TASK 3: Calculate Centroids (nearest-neighbour proxy)

sam_centroids = gdf_sam_utm.copy()
overture_centroids = gdf_ov_utm.copy()

# TODO: Replace the geometry of the polygons with their exact center points
sam_centroids['geometry'] = gdf_sam_utm.CHANGEME  # <-- Hint: Look for the centroid property
overture_centroids['geometry'] = gdf_ov_utm.CHANGEME    # <-- Hint: Look for the centroid property

# Find the nearest Overture centroid for each SAM centroid
nearest_points = gpd.sjoin_nearest(sam_centroids, overture_centroids, distance_col="distance")

# TASK 4: Calculate Mean Distance
# TODO: Calculate the average (mean) of the resulting distance column
mean_centroid_dist = nearest_points['distance'].CHANGEME # <-- Hint: standard pandas math function


In [ ]:
# Display Results
print("-" * 50)
print(f"📊 SAMGeo Performance Metrics (2016 vs 2026)")
print("-" * 50)
print(f"Total SAMGeo Area (2016):    {sam_total_area:,.1f} m²")
print(f"Total Overture Area (2026):  {ov_total_area:,.1f} m²")
print(f"Intersection Area (Overlap): {intersection_area:,.1f} m²")
print(f"SAM-only area:               {sam_only_area:,.1f} m²")
print(f"Overture-only area:          {overture_only_area:,.1f} m²")
print("-" * 50)
print(f"Precision-like consistency:  {precision:.3f}  (Are predictions actually buildings?)")
print(f"Recall-like consistency:     {recall:.3f}  (Did we find all buildings?)")
print(f"F1-like consistency:         {f1_score:.3f}")
print("-" * 50)
print(f"Mean Centroid Shift:         {mean_centroid_dist:.2f} meters")
print("-" * 50)

<div class="alert alert-success"> 

**🧠 Questions**: 
- Which mismatch type is likely to be most useful for detecting urban change?
- Which metric is most sensitive to small geometric offsets?
- If orthophoto and Overture are slightly misaligned, which metric will be most misleading?
- Which metric is most robust to noise and minor artifacts in the data?

### 9.1 (optional) Create hotspot zones

<div class="alert alert-info">

The municipality does not need a single AOI-wide score.  
It needs **sub-regions** to inspect first.

So we now:
1. create a small grid over the AOI (e.g., 100m x 100m cells), namely a fishnet,
2. measure disagreement inside each grid cell,
3. rank the cells by disagreement,
4. map the top hotspots.

</div>

<div class="alert alert-danger">

### 🚀 TODO — Identify top hotspot cells
1. Create a fishnet grid over the AOI
2. Overlay the disagreement polygons with that grid
3. Compute a disagreement score per cell
4. Rank the cells
5. Keep the **top 3** for your final recommendation

</div>

In [ ]:
def make_fishnet_from_bbox(bbox: list, cell_size_m: int, crs_projected: str) -> gpd.GeoDataFrame:
    """Create a square fishnet grid over the AOI.
    
    args:
        bbox: bounding box [min_lon, min_lat, max_lon, max_lat]
        cell_size_m: cell size in meters
        crs_projected: projected CRS string (e.g., "EPSG:32632") - must be in meters
    """
    
    min_lon, min_lat, max_lon, max_lat = bbox
    aoi = gpd.GeoDataFrame(
        {"id": [1]},
        geometry=[box(min_lon, min_lat, max_lon, max_lat)],
        crs="EPSG:4326",
    ).to_crs(crs_projected)

    minx, miny, maxx, maxy = aoi.total_bounds

    cells = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            cells.append(box(x, y, x + cell_size_m, y + cell_size_m))
            y += cell_size_m
        x += cell_size_m

    grid = gpd.GeoDataFrame({"geometry": cells}, crs=crs_projected)
    grid = gpd.overlay(grid, aoi, how="intersection")
    grid = grid.reset_index(drop=True)
    grid["cell_id"] = grid.index + 1
    return grid


In [ ]:
grid = make_fishnet_from_bbox(
    # TODO fill the values
)

# Compute disagreement area per cell
sam_only["source"] = "sam_only"
overture_only["source"] = "overture_only"
disagreement = pd.concat([sam_only, overture_only], ignore_index=True)
disagreement = gpd.GeoDataFrame(disagreement, geometry="geometry", crs=CONFIG["crs_projected"])

# Join the grid with the disagreement polygons
grid_join = gpd.overlay(grid, disagreement, how="intersection")
grid_join["piece_area_m2"] = grid_join.geometry.area

# Group by cell_id and sum the piece_area_m2
hotspots = (
    grid_join.groupby("cell_id", as_index=False)["piece_area_m2"]
    .sum()
    .rename(columns={"piece_area_m2": "disagreement_area_m2"})
)

# Calculate disagreement ratio
# TODO: Calculate the disagreement ratio by dividing the disagreement area by the cell area
grid_hotspots = grid.merge(hotspots, on="cell_id", how="left")
grid_hotspots["disagreement_area_m2"] = grid_hotspots["disagreement_area_m2"].fillna(0)
grid_hotspots["cell_area_m2"] = grid_hotspots.geometry.area
grid_hotspots["disagreement_ratio"] = # TODO calculate the ratio

grid_hotspots = grid_hotspots.sort_values("disagreement_ratio", ascending=False)
grid_hotspots.to_file(CONFIG["hotspot_file"], driver="GPKG")

grid_hotspots[["cell_id", "disagreement_area_m2", "disagreement_ratio"]].head(10)


In [ ]:
hotspots_wgs = grid_hotspots.to_crs(CONFIG["crs_geographic"])

fig = px.choropleth_mapbox(
    hotspots_wgs,
    geojson=hotspots_wgs.geometry,
    locations=hotspots_wgs.index,
    color="disagreement_ratio",
    center={
        "lat": hotspots_wgs.geometry.centroid.y.mean(),
        "lon": hotspots_wgs.geometry.centroid.x.mean(),
    },
    zoom=15,
    mapbox_style="carto-positron",
    opacity=0.6,
    hover_data=["cell_id", "disagreement_area_m2", "disagreement_ratio"],
    title="Hotspots of historical/current building disagreement",
)
fig.show()

<div class="alert alert-success"> 

**🧠 Questions**: 
- Why is a hotspot map more useful to the municipality than one AOI-wide score?
- Should the city prioritize the **largest disagreement area** or the **highest disagreement ratio**?
- What kinds of false alarms could still appear in the top-ranked cells?

---

## 10. Limits of the exercise

---

- **Temporal mismatch:** historical image vs current vector data
- **No authoritative reference:** Overture is useful, but not official cadastral truth
- **SAMGeo is a general model:** it may segment roofs and noise, not just buildings
- **Roof ≠ footprint:** image geometry and vector building geometry may differ
- **Small AOI bias:** your result may not generalise across the whole city
- **Operational ambiguity:** disagreement suggests review, not certainty

---

## 10. 👀 Peer Feedback 

---


Exchange your notebook with a classmate, compare your implementation and results, and discuss any differences, bugs, or unclear choices. After the discussion, improve your notebook based on the feedback you received.

---

## 11. 📫 Annexes

---

In [ ]:
# GEOAI Segmentation
# https://opengeoai.org/examples/building_footprints_usa/#visualize-building-footprints

import geoai

ortho_path = str(DATA_DIR / "graz_ortho_clipped.tif")
ai_buildings_path = str(OUTPUT_DIR / "geoai_buildings_centre.geojson")

# Initialize the building footprint extraction model
print("Initializing the GeoAI Building Extractor...")
extractor = geoai.BuildingFootprintExtractor()

# Extract building footprints directly to vector (Option 2 from docs)
print("Analyzing the 2016 orthophoto and extracting buildings... (This may take a minute)")
gdf_ai = extractor.process_raster(
    ortho_path,
    output_path=ai_buildings_path,
    batch_size=4,
    confidence_threshold=0.5,
    overlap=0.25,
    nms_iou_threshold=0.5,
    min_object_area=50,
    mask_threshold=0.5,
    simplify_tolerance=1.0,
)

print(f"Found {len(gdf_ai)} raw building footprints")

# Regularize the polygons (Square off the corners)

gdf_ai_reg = extractor.regularize_buildings(
    gdf=gdf_ai,
    min_area=50,
    angle_threshold=15,
    orthogonality_threshold=0.3,
    rectangularity_threshold=0.7,
)

# Ensure the CRS matches our WGS84 standard
gdf_ai_reg.set_crs(CONFIG["crs_geographic"], inplace=True, allow_override=True)

print(f"Regularized {len(gdf_ai_reg)} buildings")
gdf_ai_reg.head()

In [ ]:
# TODO: Decide if we want to keep this!

ortho_path = str(DATA_DIR / "graz_ortho_clipped.tif")
geoai_file = str(OUTPUT_DIR / "geoai_buildings_centre.geojson")

# 2. Load the GeoAI polygons
gdf_geoai_map = gpd.read_file(geoai_file)

# Calculate the center of our bounding box to focus the map
min_lon, min_lat, max_lon, max_lat = ACTIVE_BBOX
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2


# 3. Initialize the map
m_geoai = leafmap.Map(center=[center_lat, center_lon], zoom=16, height="650px")
m_geoai.clear_layers()

# 4. Add the clipped historical orthophoto as the basemap
m_geoai.add_raster(ortho_path, layer_name="Historical Orthophoto (2016)")

# 5. Add the GeoAI Polygons on top
m_geoai.add_gdf(
    gdf_geoai_map,
    layer_name="GeoAI Regularized Buildings",
    fill_colors=["#ffaa00"], # A nice visible orange/gold
    color="darkorange",
    weight=2
)

# Display the map
m_geoai

**You are at the end congrats 💪 🎉**